# Data Science Lab - Project

## Import packages and modules

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import librosa
from sklearn.pipeline import make_pipeline
from scipy.stats import skew, kurtosis
from scipy.signal import hilbert
from math import log10
import parselmouth
import noisereduce as nr
from maad import sound as suono
from maad import features as ft

c:\Users\ghiot\OneDrive\Desktop\Work in Progress\Poli\DataScienceLab\01TWZSM_DataScienceLab\virtual_environment_DSL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import matplotlib
print(matplotlib.__version__)

3.10.0


## Load data

#### Load data from csv file and first feature transformations

In [2]:
csvTotale = pd.read_csv('.././data/development.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvTotale['gender'] = csvTotale['gender'].map(lambda x: 0 if x=='male' else 1)
csvTotale['tempo'] = csvTotale['tempo'].map(lambda x: float(x[1:-1]))
csvTotale['path'] = csvTotale['path'].map(lambda x: x.split('/')[-1])

In [4]:
csvEval = pd.read_csv('.././data/evaluation.csv', header=0, index_col=0).drop(columns=['sampling_rate'])
csvEval['gender'] = csvEval['gender'].map(lambda x: 0 if x=='male' else 1)
csvEval['tempo'] = csvEval['tempo'].map(lambda x: float(x[1:-1]))
csvEval['path'] = csvEval['path'].map(lambda x: x.split('/')[-1])

In [5]:
"""(cross_val_score(HistGradientBoostingRegressor(categorical_features=csvTotale.drop(columns=['path', 'age']).dtypes == 'object'), 
                csvTotale.drop(columns=['path', 'age']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean(),
 cross_val_score(MLPRegressor(), 
                csvTotale.drop(columns=['path', 'age', 'ethnicity']), 
                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean())"""

# BASELINE = -7.24, -8.18

"(cross_val_score(HistGradientBoostingRegressor(categorical_features=csvTotale.drop(columns=['path', 'age']).dtypes == 'object'), \n                csvTotale.drop(columns=['path', 'age']), \n                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean(),\n cross_val_score(MLPRegressor(), \n                csvTotale.drop(columns=['path', 'age', 'ethnicity']), \n                csvTotale['age'], cv=10, scoring='neg_mean_absolute_error').mean())"

#### Load and clean audio files

In [6]:
def load_data(audio_files, folder_path, sample_rate):
    audio_arrays = {}
    for file_name in audio_files:
        if file_name.endswith(".wav"):
            audio_array = nr.reduce_noise(librosa.load(folder_path + file_name, sr=sample_rate)[0], sr=sample_rate, stationary=False)
            trimmed_audio_array= librosa.effects.trim(audio_array)
            audio_arrays[file_name] = trimmed_audio_array[0]+1e-10
    return audio_arrays

In [7]:
SAMPLE_RATE = 22050
audioDevelopment = load_data(csvTotale['path'], '.././data/audios_development/', SAMPLE_RATE)
audioEval = load_data(csvEval['path'], '.././data/audios_evaluation/', SAMPLE_RATE)

In [22]:
audioDevelopment = {file: librosa.load('.././data/audios_development/'+file, sr=22050)[0]+1e-10 for file in csvTotale['path']}
audioEval = {file: librosa.load('.././data/audios_evaluation/'+file, sr=22050)[0]+1e-10 for file in csvEval['path']}

## Feature extraction

In [9]:
def getMFCC(audio):
    numFcc = 35
    return pd.Series(librosa.feature.mfcc(y=audio, sr=22050, n_mfcc=numFcc).mean(axis=1), 
                     index=[f'mfcc{i}' for i in range(numFcc)])

In [10]:
def getSpectralEnergy(audio):
    S = librosa.stft(audio)
    freqs = librosa.fft_frequencies(sr=22050)
    return pd.Series([np.sum(np.abs(S[(freqs >= 250) & (freqs <= 650)])**2), np.sum(np.abs(S[(freqs >= 1000) & (freqs <= 8000)])**2)],
                        index=['spectralEnergy250-650', 'spectralEnergy1000-8000'])

In [11]:
def computeF0(audio):
    sound = parselmouth.Sound(audio)
    pitch = sound.to_pitch()
    info = str(pitch.info()).split('\n')
    return pd.Series([pitch.count_voiced_frames(), pitch.get_mean_absolute_slope(), pitch.xmax-pitch.xmin, 
                      pitch.n_frames, *[float(info[15+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 5)],
                      *[float(info[21+i].split('=')[2].lstrip().split(' =')[0].split()[0]) for i in range(0, 3)]],
                     index=['nVoicedFrames', 'meanAbsoluteSlope', 'duration', 'nFrames', 
                            'q10', 'q16', 'q50', 'q84', 'q90', '84-median', 'median-16', '90-10']
                     )

In [12]:
def computeMath(audio):
    return pd.Series([skew(audio), kurtosis(audio), np.mean(np.abs(hilbert(audio))), np.mean(np.abs(np.fft.fft(audio)))],
                     index=['skew', 'kurtosis', 'hilbertMean', 'fftMean'])

In [13]:
def computSNR(audio):
    return pd.Series([suono.temporal_snr(audio)[-1]],
                        index=['temporalSNR'])

In [14]:
def computeTemporalMedia(audio):
    return pd.Series([ft.temporal_median(audio)],
                     index=['temporalMedian'])

In [15]:
def computeAllFeatures(audio):
    return pd.Series(ft.all_temporal_features(audio, fs=22050, nperseg=256).values[0],
                     index=['sm', 'sv', 'ss', 'sk', 'Time 5%', "Time 25%", "Time 50%", "Time 75%", "Time 95%  ", 
                            "zcr", "duration_5", "duration_90"])

In [16]:
def computePeakFrequency(audio):
    return pd.Series(ft.peak_frequency(audio, fs=22050, nperseg=256, amp=True),
                     index=['peakFrequency', 'peakFrequencyAmp'])

In [17]:
def computeEntropy(audio):
    return pd.Series(ft.temporal_entropy(audio), index=['entropy'])

In [18]:
# NO BUENO JUST NOISE
def computeMaadStats(audio):
    Sxx_power, tn, fn, _ = suono.spectrogram(audio, 22050)      
    return pd.Series([ft.number_of_peaks(Sxx_power, fn=fn, nperseg=256), 
                      ft.bioacoustics_index(Sxx_power, fn=fn, flim=(100, 3000)),
                      ft.acoustic_diversity_index(Sxx_power, fn=fn, fmin=80, fmax=3000),
                      ft.acoustic_eveness_index(Sxx_power, fn=fn, fmin=80, fmax=3000),
                    # ft.roughness(Sxx_power)
                      ],
                    index=['nPeaks', 'bioIndex', 'acousticDiversity', 'acousticEvenness'])

#### Overall function

In [19]:
def computeMetrics(audios):
    return pd.DataFrame({file: pd.concat([
            getMFCC(audios[file]),
            getSpectralEnergy(audios[file]),
            computeF0(audios[file]),
            computeMath(audios[file]),
            computSNR(audios[file]),
            computeTemporalMedia(audios[file]),
            computeAllFeatures(audios[file]),
            computePeakFrequency(audios[file]),
            computeEntropy(audios[file]),   # fucking entropy drops the score
            ], axis=0)
        for file in audios}).T

In [23]:
metrics = pd.concat([computeMetrics(audioDevelopment), 
                    csvTotale.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)

metrics['log_hnr']= metrics['hnr'].map(lambda x: 20*log10(np.abs(x)))

## Grid search and model selection

#### Grid search

DEFAULT PARAMETERS\
loss='squared_error',\
quantile=None,\
learning_rate=0.1,\
max_iter=100,\
max_leaf_nodes=31,\
max_depth=None,\
min_samples_leaf=20,\
l2_regularization=0.0,\
max_features=1.0,\
max_bins=255,\
categorical_features='warn',\
monotonic_cst=None,\
interaction_cst=None,\
warm_start=False,\
early_stopping='auto',\
scoring='loss',\
validation_fraction=0.1,\
n_iter_no_change=10,\
tol=1e-07,\
verbose=0,\
random_state=None

In [27]:
RANDOM_SEED = 42
cv_fold = 20
param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_iter': [80, 100, 150, 200],
    'max_leaf_nodes': [30, 35, 40, 45],
    'l2_regularization': [0.0, 0.5, 1.0], 
    'max_depth': [None, 20, 30],
    'min_samples_leaf': [15, 20, 25],
    'tol': [1e-08, 1e-07, 1e-06]
}

In [ ]:
grid_search = GridSearchCV(HistGradientBoostingRegressor(random_state=RANDOM_SEED), param_grid, scoring='neg_root_mean_squared_error', cv=cv_fold,return_train_score=True, n_jobs=2).fit(metrics, csvTotale['age'])
best_hgbr = grid_search.best_estimator_

c:\Users\ghiot\OneDrive\Desktop\Work in Progress\Poli\DataScienceLab\01TWZSM_DataScienceLab\virtual_environment_DSL\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ghiot\OneDrive\Desktop\Work in Progress\Poli\DataScienceLab\01TWZSM_DataScienceLab\virtual_environment_DSL\Lib\site-packages\joblib\externals\loky\backend\context.py", line 282, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [30]:
rand_grid_search = RandomizedSearchCV(HistGradientBoostingRegressor(random_state=RANDOM_SEED), param_grid, scoring='neg_root_mean_squared_error', cv=cv_fold, n_iter=150, return_train_score=True, n_jobs=2).fit(metrics, csvTotale['age'])
rand_best_hgbr = rand_grid_search.best_estimator_

In [32]:
results = pd.DataFrame(rand_grid_search.cv_results_)
scoreInterest= 'mean_test_score' # mean_test_score, mean_train_score, mean_fit_time
interestingColumns = results[[col for col in results.columns if 'param_' in col]+[scoreInterest]]
print('BEST HIPERPARAMETERS')
max_elem_row=interestingColumns[scoreInterest].idxmax()
display(pd.DataFrame(interestingColumns.iloc[max_elem_row]))  #.transpose())

BEST HIPERPARAMETERS


,108
param_tol,0.0
param_min_samples_leaf,15
param_max_leaf_nodes,30
param_max_iter,200
param_max_depth,30
param_learning_rate,0.05
param_l2_regularization,0.0
mean_test_score,-9.677748


#### Defaul parameters

In [26]:
cross_val_score(HistGradientBoostingRegressor(), metrics.drop(columns='entropy'),csvTotale['age'], cv=20, scoring='neg_root_mean_squared_error').mean()

np.float64(-9.846578249261853)

## Feature correlation with age

In [32]:
for item, imp in sorted(zip(list(metrics.columns)+['age'], 
                            pd.concat([metrics, csvTotale.set_index(csvTotale['path'])['age']], axis=1).corr('spearman')['age']), key=lambda x: abs(x[1]), reverse=True):
    print(f'{item}: {imp}')

age: 1.0
nFrames: 0.602208425023726
duration: 0.6021939399131905
Time 95%  : 0.5992034454587636
duration_90: 0.5966730708595309
Time 75%: 0.5894010167208369
duration_5: 0.5861552080799808
Time 50%: 0.5755312124515707
nVoicedFrames: 0.55795268213888
Time 25%: 0.5360900192790657
hnr: -0.5326292343962696
log_hnr: 0.5320784492500874
spectralEnergy250-650: 0.5213179481891375
Time 5%: 0.4924232038961484
sm: 0.48387186146167255
fftMean: 0.48371923800568034
spectralEnergy1000-8000: 0.47677967673179394
max_pitch: 0.4694639426920942
temporalSNR: 0.4647672034900796
min_pitch: -0.46452284591811516
mfcc6: -0.4427071277985795
zcr: 0.43274720129527433
kurtosis: 0.3675260831570178
sk: 0.3675192405478306
mean_pitch: 0.36619218880953386
mfcc1: -0.357953723981469
ss: -0.35646826357014166
skew: -0.3564670008395054
jitter: 0.34920179676737306
mfcc3: -0.33569512236044957
mfcc9: -0.31722287761622775
mfcc33: -0.30582535276540435
mfcc2: -0.3057006608539275
mfcc13: -0.3004749341866707
mfcc17: -0.300095824200978

## Test on evaluation data

In [33]:
metricsEval = pd.concat([computeMetrics(audioEval), 
                    csvEval.set_index('path')[['hnr', 'shimmer', 'jitter', 'gender', 'max_pitch', 'mean_pitch', 'min_pitch']]], axis=1)
metricsEval['log_hnr']= metricsEval['hnr'].map(lambda x: 20*log10(np.abs(x)))

In [34]:
def roundData(eval_pred):
    return eval_pred.map(lambda x: int(x) if x-int(x)<0.25 else x).map(lambda x: int(x)+0.5 if 0.25<=x-int(x)<0.75 else x).map(lambda x: int(x)+1 if 0.75<=x-int(x) else x)

In [37]:
#gest_hgbr = grid_search.best_estimator_
evaluation_pred = pd.Series(rand_best_hgbr.predict(metricsEval),name='Predicted',index=csvEval.index)
rounded_eval_pred = roundData(evaluation_pred)
rounded_eval_pred.to_csv('outputs/pred_prova.csv')